<a href="https://colab.research.google.com/github/Haryapandhita/ITAI_ML_FirstRepo_MourinhoHernanto/blob/main/Mid_Term_EDA(FinalDraft)ITAI1371Section19179.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

-----------
# **Midterm Project** – Data Cleaning & Preparation for Machine Learning

# **Dataset**: CTU-IoT-Malware-Capture-8-1conn

# **Objectives**: Prepare raw network traffic data for ML by performing EDA, cleaning, feature engineering, encoding, scaling, and export.
------

In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# For plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'



In [ ]:
# =========================
# SETUP: CREATE FIGURES FOLDER (Mourinho)
# =========================
import os
os.makedirs('figures', exist_ok=True)
print("Folder '/figures' is ready.")

In [ ]:
# =========================
# 2. LOAD DATASET
# =========================
df = pd.read_csv("/content/CTU-IoT-Malware-Capture-8-1conn_Use_this_one-1 (1).csv")

# Quick look
print("RAW DATA SAMPLE:")
display(df.head())

print("\nDataset shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())


In [ ]:
# =========================
# EDA 1: MISSING VALUE VISUALIZATION (RAW DATA) (Mourinho)
# =========================
import numpy as np

# Create a temporary copy to safely replace "-" with NaN for visualization
temp_df = df.copy()
temp_df.replace("-", np.nan, inplace=True)

plt.figure(figsize=(12, 6))
sns.heatmap(temp_df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title("Visual Audit: Missing Values in Raw Dataset")
plt.savefig("figures/missing_value_heatmap.png")
plt.show()

print("Heatmap saved to /figures/missing_value_heatmap.png")

In [ ]:
# =========================
# 3. INITIAL CLEANING
# =========================

# Replace "-" with NaN
df.replace("-", np.nan, inplace=True)

# Drop unneeded identifier columns
df.drop(columns=["uid", "id.orig_h", "id.resp_h", "id.resp_p", "history"], errors="ignore", inplace=True)


In [ ]:
# =========================
# 4. TRAIN/TEST SPLIT
# =========================
train_df, test_df = train_test_split(df, test_size=0.30, random_state=42)

print("Training size:", train_df.shape)
print("Testing size:", test_df.shape)


In [ ]:
# =========================
# 5. CONVERT NUMERIC COLUMNS
# =========================
cols_to_fix = ["duration", "orig_bytes", "orig_pkts"]

for col in cols_to_fix:
   train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
   test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

# Fill missing values with median from training set
train_medians = train_df[cols_to_fix].median()
for col in cols_to_fix:
   train_df[col] = train_df[col].fillna(train_medians[col])
   test_df[col] = test_df[col].fillna(train_medians[col])


In [ ]:
# =========================
# EDA 2: FEATURE DISTRIBUTION (TRAINING DATA ONLY) (Mourinho)
# =========================
plt.figure(figsize=(10, 5))
sns.histplot(train_df["duration"], kde=True, color="teal")
plt.title("Distribution of Connection Duration (Training Set)")
plt.xlabel("Duration (seconds)")
plt.savefig("figures/duration_distribution.png")
plt.show()

print("Histogram saved to /figures/duration_distribution.png")

In [ ]:
# =========================
# 6. FEATURE ENGINEERING
# =========================
train_df["bytes_per_packet"] = train_df["orig_bytes"] / (train_df["orig_pkts"] + 1)
test_df["bytes_per_packet"] = test_df["orig_bytes"] / (test_df["orig_pkts"] + 1)

train_df["packet_density"] = train_df["orig_pkts"] / (train_df["duration"] + 1)
test_df["packet_density"] = test_df["orig_pkts"] / (test_df["duration"] + 1)


In [ ]:
# =========================
# 7. ONE-HOT ENCODING CATEGORICAL FEATURES
# =========================
# Convert categorical columns to dummies
train_df = pd.get_dummies(train_df, drop_first=True)
test_df = pd.get_dummies(test_df, drop_first=True)

# Align test columns to training columns
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)


In [ ]:
# =========================
# 9. SPLIT FEATURES AND TARGET
# =========================
# Print columns to identify the correct target column name after one-hot encoding
print("Columns in train_df before splitting:")
print(train_df.columns)

# Based on the printed columns, the target column is 'label_Malicious'
X_train = train_df.drop("label_Malicious", axis=1)
y_train = train_df["label_Malicious"]

X_test = test_df.drop("label_Malicious", axis=1)
y_test = test_df["label_Malicious"]

# Verify shapes and label distribution
print("Training features shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("Test features shape:", X_test.shape)
print("Test target shape:", y_test.shape)

print("\nTraining label distribution:\n", y_train.value_counts())
print("\nTest label distribution:\n", y_test.value_counts())

In [ ]:
# =========================
# EDA 3: TARGET DISTRIBUTION (TRAINING DATA ONLY) (Mourinho)
# =========================
plt.figure(figsize=(6, 4))
y_train.value_counts().plot(kind='bar', color=['salmon', 'skyblue'])
plt.title("Label Distribution: Malicious vs Benign (Training Set)")
plt.xticks([0, 1], ['Malicious (True)', 'Benign (False)'], rotation=0)
plt.savefig("figures/target_distribution.png")
plt.show()

print("Bar chart saved to /figures/target_distribution.png")

In [ ]:
# =========================
# 10. SCALE FEATURES
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Quick sample
display(X_train_scaled.head())


In [ ]:
# =========================
# 11. EXPORT CLEAN DATASETS
# =========================
train_df_final = pd.concat([X_train_scaled, y_train.reset_index(drop=True)], axis=1)
test_df_final = pd.concat([X_test_scaled, y_test.reset_index(drop=True)], axis=1)

train_df_final.to_csv("clean_training_dataset_scaled.csv", index=False)
test_df_final.to_csv("clean_testing_dataset_scaled.csv", index=False)


## Reflections on Data Preparation

During this project, several challenges were addressed to ensure a robust, ML-ready dataset:

**1. Avoiding Data Leakage**  
- Early EDA was initially performed before splitting the dataset, which risks data leakage.  
- All cleaning, missing value handling, and feature engineering were subsequently applied **using the training set only**, with test data processed based on training parameters.  

**2. Accidental Data Alteration**  
- Some initial commands unintentionally modified columns, particularly categorical ones.  
- Transformations were reordered and controlled to preserve original features and maintain reproducibility.  

**3. Median over Mean for Missing Values**  
- Numeric features such as `duration` and `orig_bytes` were filled with the **median** instead of the mean.  
- Network traffic data is often **skewed**: a few very long or large connections could pull the mean away from typical values.  
- Using the median ensures the imputed values reflect the central tendency of most connections.  

**4. Categorical Audit Prior to One-Hot Encoding**  
- Unique value counts were examined before encoding to verify the distribution of network protocols, services, and connection states.  
- This step ensured that rare categories were accounted for and that the binary sparse matrix used for ML models accurately reflected the data.  

**5. Lessons Learned**  
- Maintaining a clear pipeline (**clean → split → feature engineer → encode → scale**) prevents errors from propagating.  
- Documenting each preprocessing decision improves reproducibility and interpretability.  
- Visualizing categorical distributions before and after transformation helps confirm data integrity.

**6. Feature Engineering Rationale**  
- **Bytes per Packet:** Calculated as `orig_bytes / (orig_pkts + 1)` to measure the average payload size per packet, giving insight into traffic intensity per connection.  
- **Packet Density:** Calculated as `orig_pkts / (duration + 1)` to capture how rapidly packets are transmitted over time, highlighting unusual bursts typical of malicious activity.  
- These engineered features provide meaningful signals for ML models beyond raw counts and help differentiate benign from malicious network behavior.

## Best Practices Followed

- **Stepwise, reproducible pipeline:** Cleaning → Train/Test Split → Feature Engineering → Encoding → Scaling → Export, ensuring no accidental data leakage.  
- **Data integrity audits:** Checked for missing values, verified categorical distributions, and confirmed feature ranges before and after transformations.  
- **Robust statistical choices:** Median imputation for skewed numeric data, which prevents outliers from distorting the central tendency.  
- **Meaningful feature engineering:** Created features (bytes per packet, packet density) that provide additional insight for ML models.  
- **Documentation and transparency:** Each preprocessing decision is clearly explained to ensure reproducibility and interpretability, demonstrating good data science practice.